# Low-level multi-seed template

This notebook exposes the individual pieces instead of using `benchmark_method`. Use it when you need custom control over training, caching, evaluation, reporting, or per-seed artifacts. The integration method is still user-owned.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "git+https://github.com/amirhossein-alishahi/scRareBench_.git@v0.10.5"])
from scrarebench.runtime import setup_runtime
setup_runtime(quiet=False)


In [ ]:
from pathlib import Path
from scrarebench import load_dataset, dataset_info, normalize_method_seeds
adata = load_dataset(0)
info = dataset_info(adata)
METHOD_SEEDS = normalize_method_seeds([42, 123, 2026])
BENCHMARK_SEED = 42
OUT = Path("/content/scrarebench_low_level_multiseed")


## User method hook

Replace this with your method. The placeholder intentionally raises until you provide a real integration implementation.


In [ ]:
def run_user_method(adata_for_method, seed):
    raise NotImplementedError("Implement your batch-effect-removal method and return its latent representation.")


In [ ]:
from scrarebench import benchmark_latent, finalize_multiseed_delivery
reports = []
bundles = {}
for seed in METHOD_SEEDS:
    method_adata = adata.copy()
    latent = run_user_method(method_adata, seed)
    result = benchmark_latent(
        adata.copy(), latent, method="MyMethod",
        barcodes=method_adata.obs_names,
        output_dir=OUT / f"seed_{seed}",
        method_seed=seed,
        method_config={"seed": seed},
        expected_seeds=METHOD_SEEDS,
        config={"random_state": BENCHMARK_SEED},
    )
    reports.append(result.interactive_report_path)
    bundles[seed] = result.bundle_path

final = finalize_multiseed_delivery(
    reports, OUT / "multi_seed",
    method_name="MyMethod",
    dataset_key=info.get("dataset_key") or info.get("name") or "dataset",
    expected_seeds=METHOD_SEEDS,
    bundles_by_seed=bundles,
)
print(final)
